# InsureAI — Module 2: Deep Learning (ANN on Tabular Data)
### Capstone Project: End-to-End AI Assistant for Insurance

---
### Objectives:
- Build Artificial Neural Networks (ANN) with 2–3 hidden layers for tabular insurance data.
- Premium Prediction (Regression): ReLU activations, linear output, Adam optimizer, MSE loss, Dropout regularization, EarlyStopping callback.
- Fraud Detection (Classification): ReLU activations, sigmoid output, binary crossentropy loss, Dropout, EarlyStopping.
- Plot training vs validation loss curves across epochs to visually diagnose overfitting.
- Build comparison tables directly evaluating ANN performance against Module 1 classical ML models on the exact same test splits.


In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score, f1_score, roc_auc_score
import joblib

print(f'TensorFlow Version: {tf.__version__}')


## 1. ANN Regressor for Premium Prediction


In [ ]:
# Load preprocessed tabular data
df_prem = pd.read_csv('../data/insurance_premium.csv').drop(columns=['customer_id'])
df_prem['bmi'] = df_prem['bmi'].fillna(df_prem['bmi'].median())
df_prem['annual_income_inr'] = df_prem['annual_income_inr'].fillna(df_prem['annual_income_inr'].median())

cat_cols = ['gender', 'smoker', 'region', 'occupation', 'exercise_frequency', 'alcohol_consumption', 'medical_history', 'family_medical_history']
num_cols = ['age', 'bmi', 'children', 'annual_income_inr']

enc = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
scaler = StandardScaler()

X = np.hstack([scaler.fit_transform(df_prem[num_cols]), enc.fit_transform(df_prem[cat_cols])])
y = df_prem['annual_premium_inr'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build ANN Regressor Architecture
ann_reg = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

ann_reg.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

history_reg = ann_reg.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=35,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Plot Training vs Validation Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(history_reg.history['loss'], label='Training Loss (MSE)', color='#2b5c8f', lw=2)
plt.plot(history_reg.history['val_loss'], label='Validation Loss (MSE)', color='#d9534f', lw=2)
plt.title('ANN Training vs Validation Loss — Premium Regression')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Evaluate ANN Regressor vs ML
preds_ann = ann_reg.predict(X_test).flatten()
r2 = r2_score(y_test, preds_ann)
mae = mean_absolute_error(y_test, preds_ann)
rmse = np.sqrt(mean_squared_error(y_test, preds_ann))

print(f'ANN Performance: R2 = {r2:.4f}, MAE = INR {mae:,.2f}, RMSE = INR {rmse:,.2f}')

# Comparative Results Table
comp_reg = pd.DataFrame([
    {'Model': 'Linear Regression', 'R2 Score': 0.8982, 'MAE (INR)': 2516.58, 'RMSE (INR)': 3200.35, 'Training Speed': 'Very Fast', 'Interpretability': 'High'},
    {'Model': 'Random Forest', 'R2 Score': 0.9164, 'MAE (INR)': 2313.67, 'RMSE (INR)': 2899.36, 'Training Speed': 'Fast', 'Interpretability': 'Medium'},
    {'Model': 'XGBoost', 'R2 Score': 0.9294, 'MAE (INR)': 2125.05, 'RMSE (INR)': 2664.75, 'Training Speed': 'Fast', 'Interpretability': 'Medium'},
    {'Model': 'ANN (Deep Learning)', 'R2 Score': round(r2, 4), 'MAE (INR)': round(mae, 2), 'RMSE (INR)': round(rmse, 2), 'Training Speed': 'Slower', 'Interpretability': 'Low'}
])
display(comp_reg)
